In [2]:
print("Hello woeld")

Hello woeld


## Usando o pandas para analisar o arquivo
---
* Caminho: C:\Users\joao.victor\MSGÁS\GEOP - Documentos\2026\Manutenção - 2026

In [3]:
#instala as bibliotecas

#!pip install pandas
#!pip install openpyxl
#!pip install dotenv

  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)

   ---------------------------------------- 2/2 [dotenv]



In [1]:
#importa bibliotecas
import pandas as pd
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

camihho = os.getenv("FILE_BASE_DATA")
try:
    df = pd.read_excel(camihho, sheet_name='IFS_TASK_CLOCKING')
    print("Carregado dataframe")
except Exception as e:
    print('Erro ao ler o excel', e)

Carregado dataframe


## Tratamento de dados
* Apenas colunas, Nome em PT-BR - Nome em ING-IFSfrescos
   * N° Tarefa - TASK_SEQ
   * Descr da Tarefa - TASK_DESCRIPTION
   * Categoria de Registro - CLOCKING_CATEGORY (Viagem ou serviço)
   * Tipo de Reg de Horas - CLOCKING_TYPE (Registro de saída ou de entrada)
   * Org de Manut - ORGANIZATION_ID
   * Horário de Início - START_TIME
   * Hora Parada - STOP_TIME
   * Horas Trab - WORK_HOURS
   * ID Recurso - EMPLOYEE_ID
* Limpa linhas que estão incompletas
* Modifica os dtypes das colunas
*

In [3]:
essential_columns = [
  "TASK_SEQ",
  "TASK_DESCRIPTION",
  "CLOCKING_CATEGORY",
  "CLOCKING_TYPE",
  "START_TIME",
  "STOP_TIME",
  "WORK_HOURS",
  "ORGANIZATION_ID",
  "EMPLOYEE_ID"
]
df_clean = df[essential_columns]

In [4]:
# mudando os nomes
df_clean.rename(columns={"TASK_SEQ" : "ID_tarefa",
  "TASK_DESCRIPTION": "Descricao",
  "CLOCKING_CATEGORY": "Tipo_temporal",
  "CLOCKING_TYPE": "Valida_registro",
  "START_TIME": "Tempo_inicio",
  "STOP_TIME": "Tempo_fim",
  "WORK_HOURS": "Horas_trabalhadas",
  "ORGANIZATION_ID": "ORG_manut",
  "EMPLOYEE_ID": "TOM"}, inplace=True)

In [5]:
#Retirando dados que estão em execucao
df_clean.dropna(axis=0, how='any', inplace=True)

In [ ]:
#Retira serviços de terceiros ORG MANUT -> != MCGR, OCGR, TLG


In [6]:
# Formata os tempo_inicio e tempo_fim
#       função de formatação
def formata_data(d):
    #Variaveis
    r = str(d) #Variavel de retorno
    month_number = {
        "jan": "01",
        "feb": "02",
        "mar": "03",
        "apr": "04",
        "may": "05",
        "jun": "06",
        "jul": "07",
        "aug": "08",
        "sep": "09",
        "oct": "10",
        "nov": "11",
        "dec": "12"
    } # Dicionario conversor de mes de nome para numero
    r = r.strip().lower().split() #tirando espaços e deoxando tudo minusculo
    day = r[1].replace(",", " ").strip()
    month = month_number[r[0]]
    year = r[2][:4]
    hour = pd.to_timedelta(r[3]) + pd.to_timedelta("12:00:00") if r[4] == "pm" else pd.to_timedelta(r[3]) #horas "brasileiras"
    hour = str(hour)[-8:] # Transform o dado em string
    #Formato antes: May 5, 2026, 2:37:30 PM
    #Formato que deve ficar 05/05/2026 02:37:30
    return day + "/" + month + "/" + year + " " + hour
#       aplica função de formatação
df_clean["Tempo_inicio"] = df_clean["Tempo_inicio"].apply(formata_data)
df_clean["Tempo_fim"] = df_clean["Tempo_fim"].apply(formata_data)

In [7]:
# Formata colunas de tempo para deltatime
df_clean["Tempo_inicio"] = pd.to_datetime(df_clean["Tempo_inicio"], format="%d/%m/%Y %H:%M:%S")
df_clean["Tempo_fim"] = pd.to_datetime(df_clean["Tempo_fim"], format="%d/%m/%Y %H:%M:%S")

In [8]:
filtro_temporal = df_clean["Tempo_inicio"].between(pd.to_datetime("04/01/2026"), pd.to_datetime("04/30/2026"))
df_periodo = df_clean[filtro_temporal]

## Analisando dados
* Novo df
|N° da tarefa|Decrição da tarefa|
|-------|----------|---------|


In [9]:
df_periodo

,ID_tarefa,Descricao,Tipo_temporal,Valida_registro,Tempo_inicio,Tempo_fim,Horas_trabalhadas,ORG_manut,TOM
2776,204432,Registrar inspeão de odorante,Serviço,Saída Registrada,2026-04-29 08:53:46,2026-04-29 10:01:01,1.1208,OCGR,ADEMIR
2777,204430,ACOMPANHAMENTO TÉCNICO - RUA GOIÁS ENTRE AS RU...,Serviço,Saída Registrada,2026-04-29 08:38:02,2026-04-29 13:05:00,4.4494,OCGR,ELVIS
2778,204430,ACOMPANHAMENTO TÉCNICO - RUA GOIÁS ENTRE AS RU...,Viagem,Saída Registrada,2026-04-29 08:05:21,2026-04-29 08:38:02,0.5447,OCGR,ELVIS
2780,204427,GREEN & STEAK (JARDIM CENTRAL) – Preencher med...,Serviço,Saída Registrada,2026-04-29 15:20:43,2026-04-29 16:20:27,0.9956,OCGR,ADEMIR
2781,204426,DONAH SUSHI – Preencher medidas do objeto e re...,Serviço,Saída Registrada,2026-04-29 15:20:41,2026-04-29 16:45:30,1.4136,OCGR,ADEMIR
...,...,...,...,...,...,...,...,...,...
12595,195935,Acompanhar - RETRABALHO - Instalar manilha na ...,Serviço,Saída Registrada,2026-04-10 11:01:25,2026-04-10 11:02:41,0.0211,MCGR,GILMARB
13438,195267,RETRABALHO - Instalar manilha na cx. vlv. RSYM...,Serviço,Saída Registrada,2026-04-13 10:01:05,2026-04-13 10:09:36,0.1419,NAVE,NAVEDRILL
13446,195259,MELHORIA - Inst. de laje de cx. de vv. 30 cm -...,Serviço,Saída Registrada,2026-04-14 08:39:00,2026-04-14 11:28:42,2.8283,NAVE,NAVEDRILL
15005,194070,"Preventiva est. peq. PS= 1,0 kgf/cm²",Serviço,Saída Registrada,2026-04-17 16:09:10,2026-04-17 16:11:43,0.0425,MCGR,GILMAR
